# Automotive Data Mapper - MVP: Exploratory Data Analysis (EDA)

Date: 2026-08-06  
Author: Luis Renteria Lezano  
[LinkedIn](https://www.linkedin.com/in/renteria-luis) | [GitHub](https://github.com/renteria-luis) | [Portfolio](https://luisrenteria.me)

## Executive Summary

- Goal: explore the raw automotive service-record feeds before mapping them into one canonical schema, starting with the data types and formatting problems found in the Shop A feed. This notebook is the first step of the exploratory phase of a vehicle history record (VHR) style data-mapping project.
- **Sources:** Three synthetic vehicle service feeds representing an **independent repair shop**, a **dealership management system**, and a **fleet maintenance provider**:
  - `shop_a`: CSV
  - `dealer_b`: XML
  - `fleet_c`: JSON
- **Data:** [`../data/raw/`](https://github.com/renteria-luis/automotive-data-mapper/tree/main/data/raw)
- **Data dictionary:** [`../docs/DATA_DICTIONARY.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/DATA_DICTIONARY.md)
- **Sample data documentation:** [`../docs/SAMPLE_DATA.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/SAMPLE_DATA.md)

## 0. Reproducibility & Environment Setup

Imports, routes, versions.

In [1]:
import pandas as pd

# file paths
shop_a_file_path = '../data/raw/shop_a/service_records_20260731.csv'
dealer_b_file_path = '../data/raw/dealer_b/ProcessRepairOrder_20260731.xml'
fleet_c_file_path = '../data/raw/fleet_c/maintenance_events_2026-07.json'

## 1. What arrives
The three files, their size, and their shape. No transformations will be applied yet. Each file type has its own problem:
- `csv`: shifts columns
- `xml`: returns empty
- `json`: changes the type

I create a copy of each file for the transformations and leave the original raw file untouched, that way we can compare before/after.

In [2]:
shop_a_raw = pd.read_csv(shop_a_file_path)
# dealer_b_raw = pd.read_xml(dealer_b_file_path)
# fleet_c_raw = pd.read_json(fleet_c_file_path)
shop_a_raw.shape

(42, 24)

In [3]:
shop_a = shop_a_raw.copy()
# dealer_b = dealer_b_raw.copy()
# fleet_c = fleet_c_raw.copy()

## 2. Reading shop_a: the delimiter trap
### 2.1 Raw look
I verify the dialect of the raw file before relying on the parser. The delimiter may not be a comma, quotes may be escaped by doubling them or using a backslash, line breaks may be `\r\n`, etc.
>`read_csv` uses default settings, so if the file does not follow them, it may **fail silently**.

In [4]:
# "rb": read binary
with open(shop_a_file_path, "rb") as f:
    # raw = f.read() # reads all
    # read til the first line break (header)
    header = f.readline()
    # # read the first 3 records
    first_record = f.readline()
    second_record = f.readline()
    third_record = f.readline()

print(f'> HEADER:\n {header}')
print(f'> FIRST RECORD:\n {first_record}')
print(f'> SECOND RECORD:\n {second_record}')
print(f'> THIRD RECORD:\n {third_record}')

> HEADER:
 b'VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,MAKE,MODEL,MODEL_YEAR,PLATE,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL\r\n'
> FIRST RECORD:
 b'1FTFW1E50KFA12345,10/10/2019,10/10/2019,"21,000",KM,184200,"Lube oil and filter, 5W30 synthetic","LUBE OIL AND FILTER, 5W30 SYNTHETIC",OIL FILTER,1,FORD,F-150,2019,CJKT 421,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca\r\n'
> SECOND RECORD:
 b'1FTFW1E50KFA12345,06/03/2020,06/04/2020,30733,KM,184203,Replace front brake pads and machine rotors,REPLACE FRONT BRAKE PADS AND MACHINE ROT,BRAKE PAD SET,2,FORD,F-150,2019,,,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca\r\n'
> THIRD RECORD:
 b'2T1BURHE4JC021345,07/31/2019,08/02/2019,33255,KM,184

- [X] It starts directly with `b'1FTW...`, which means there is no BOM.
- [X] The line ending is `\r\n`.
- [X] Commas are used as the delimiter.
- [X] Quotes are used for quoted fields. They are applied when the text contains a comma, which prevents the parser from splitting that record.
- [X] All data appears to be ASCII, so UTF-8 should be valid. This will be verified in the next cell.

In [5]:
with open(shop_a_file_path, "rb") as f:
    raw = f.read()

try:
    raw.decode('utf-8')
    print('Valid UTF-8, read_csv assumes defaults safely')
except UnicodeDecodeError:
    print('Not valid UTF-8')

Valid UTF-8, read_csv assumes defaults safely


In [6]:
shop_a.head(3)

,VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,...,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL
0,1FTFW1E50KFA12345,10/10/2019,10/10/2019,"21,000",KM,184200,"Lube oil and filter, 5W30 synthetic","LUBE OIL AND FILTER, 5W30 SYNTHETIC",OIL FILTER,1.0,...,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca
1,1FTFW1E50KFA12345,06/03/2020,06/04/2020,30733,KM,184203,Replace front brake pads and machine rotors,REPLACE FRONT BRAKE PADS AND MACHINE ROT,BRAKE PAD SET,2.0,...,NaN,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca
2,2T1BURHE4JC021345,07/31/2019,08/02/2019,33255,KM,184206,"Rotate tires, adjust pressures","ROTATE TIRES, ADJUST PRESSURES",TIRE,4.0,...,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service Ltd,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca


The first record shows that `MILEAGE` is `21,000`. Because it contains a comma, it is parsed as a `str`, and when the data is split by commas, the first row ends up with 25 fields instead of 24.

### 2.2 Shape and dtypes

In [7]:
print(f'> The shape of shop_a file is: {shop_a.shape}')
shop_a.dtypes

> The shape of shop_a file is: (42, 24)


VIN                       object
RO_OPEN_DATE              object
RO_CLOSE_DATE             object
MILEAGE                   object
ODOMETER_MEASURE          object
RO_INVOICE_NUMBER          int64
SERVICE_DESCRIPTION       object
LABOR_DESCRIPTION         object
PART_NAME_DESCRIPTION     object
PART_QUANTITY            float64
MAKE                      object
MODEL                     object
MODEL_YEAR               float64
PLATE                     object
PLATE_STATE               object
MANAGEMENT_SYSTEM         object
LOCATION_ID               object
LOCATION_NAME             object
ADDRESS                   object
CITY                      object
STATE                     object
POSTAL_CODE               object
PHONE                      int64
URL                       object
dtype: object

This file has **42 records**. `MILEAGE` should be numeric, but it is `object`.

`MODEL_YEAR` and `PART_QUANTITY` are `float64`, but they should be `int64`. This indicates the presence of `NaN`, since when Pandas reads a CSV, it assigns a single data type to each column based on the values it contains. If a column contains integers and a `NaN`, Pandas converts the entire column to `float64` because `NaN` cannot be stored in the standard `int64` type.
> However, `NaN` can be handled using the nullable integer `Int64` data type, which is a potential solution.

On the other hand, if a column contains integers and even a single `str`, Pandas converts the entire column to `object`. This means Pandas could not assign a single consistent data type to the column, so the values are generally stored as Python objects. In this case, the strings remain `str`, while `NaN` values are represented as `float`

### 2.3 Column exploration
`.dtypes` does not tell us what is actually inside the column, so to verify what we mentioned earlier about a single value affecting the entire column, we inspect the values individually.

In [8]:
shop_a['MILEAGE'].map(type).value_counts()

MILEAGE
<class 'str'>      41
<class 'float'>     1
Name: count, dtype: int64

By counting the types of values in `MILEAGE`, we confirm that `read_csv` converted everything to `str`, with the only `float` being the `NaN`, as mentioned earlier. This shows that simply using `.dtype` does not tell us the actual type of a specific value, which is why we will use **Pydantic**.

### 2.4 First look: what the dtypes hde
We will convert the entire `MILEAGE` column to `Int64`, while verifying that no data was corrupted by counting the `NaN` values before and after the conversion.

In [9]:
# NaN before transformation
print(f"> NaN count before transformation: {shop_a['MILEAGE'].isna().sum()}")

# clean commas
shop_a['MILEAGE'] = shop_a['MILEAGE'].str.replace(',', '')

# data transformation
shop_a['MILEAGE'] = shop_a['MILEAGE'].astype('Int64')

# NaN counts after transformation
print(f"> NaN count after transformation: {shop_a['MILEAGE'].isna().sum()}")

> NaN count before transformation: 1
> NaN count after transformation: 1


In [10]:
shop_a['MILEAGE'].map(type).value_counts()

MILEAGE
<class 'float'>    42
Name: count, dtype: int64

The number of `NaN` values before and after the transformation is the same, so the conversion did not create any new null values. We can also see that all the values are now numeric. However, `.map(type)` is not the ideal way to check the data type of individual values, that is why we are getting all `float` in the output before. It is better to use `.dtype` directly:

In [11]:
shop_a['MILEAGE'].dtype

Int64Dtype()

> **Rule:** if the `dtype` gives us a guarantee, `dtype` is the truth. If the `dtype` is **`object`**, we need to inspect it

*This column was fixed here as a worked example; the rest of the transformations are defined after the mapping dictionary.*

### 2.6 Columns used for mapping

An inventory of the columns that change the mapping.

We need to ask two questions about a column:
- **PROFILING**
    - **A. Does what I discovered during profiling change how I need to process the data?** Yes: it changes the pipeline design | No: document it for later.  
    All of these findings affect how the pipeline logic will be designed. If `NaN` values are found, we need to decide whether to reject the record, ignore the missing value, or use a fallback, such as checking another field in the same record or another source, and then making the decision again.
    The same applies to duplicate values when a field is expected to be a unique identifier. For example, if the invoice number is not unique, it cannot be used as a unique identifier by itself. In that case, we could combine `invoice + VIN` to create a unique identifier for that feed, also if different `units` are found like KM and MI, we'll decide to use KM and make a conversion of the MI.
- **VALIDATION**
    - **B. Does the content determine which records pass?**  
  These are the rules applied to each field after profiling. They are specific rules that determine which records ultimately pass into the canonical schema. For example: VIN contains a `NaN`, so the record is rejected. VIN does not comply with ISO 3779, so the record is rejected. And so on.

>For VIN, for example, finding that there are `NaN` values is **A. Profiling**, but deciding to reject those `NaN` values is **B. Validation**.

In [12]:
# profiling odometer measure
shop_a['ODOMETER_MEASURE'].value_counts()

ODOMETER_MEASURE
KM    36
MI     6
Name: count, dtype: int64

In [13]:
# profiling locaiton name
shop_a['LOCATION_NAME'].value_counts()

LOCATION_NAME
RIVERSIDE AUTO SERVICE        15
Riverside Auto Service        14
Riverside Auto Service Ltd    13
Name: count, dtype: int64

In [14]:
# profiling invoice number
shop_a['RO_INVOICE_NUMBER'].value_counts().sort_values(ascending=False).head(5)

RO_INVOICE_NUMBER
184302    2
184203    1
184206    1
184209    1
184212    1
Name: count, dtype: int64

In [15]:
# verifying repeated (duplicated? or 2 lines in the same order?) 
shop_a[shop_a['RO_INVOICE_NUMBER'] == 184302]
# duplicated -> E009

,VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,...,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL
34,1FA6P8TH2J5142608,01/29/2019,01/30/2019,65220,KM,184302,Weld exhaust flex pipe,WELD EXHAUST FLEX PIPE,CONVERTER,1.0,...,ON,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca
35,1FA6P8TH2J5142608,01/29/2019,01/30/2019,65220,KM,184302,Weld exhaust flex pipe,WELD EXHAUST FLEX PIPE,CONVERTER,1.0,...,ON,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca


### 2.7 Mapping coverage
Canonical schema fields: source_id, source_record_id, vin, vin_valid, event_date, odometer_km, odometer_source_unit, raw_description, normalized_description, provider_name, provider_city, provider_province, ingested_at

In [16]:
# this dict will exist in ../src, and will be called in 2.6
shop_a_map = {
    'VIN': 'vin',
    'RO_OPEN_DATE': 'event_date',
    'RO_CLOSE_DATE': None,
    'MILEAGE': 'odometer_km',
    'ODOMETER_MEASURE': 'odometer_source_unit',
    'RO_INVOICE_NUMBER': 'source_record_id',
    'SERVICE_DESCRIPTION': 'raw_description',
    'LABOR_DESCRIPTION': None,
    'PART_NAME_DESCRIPTION': None,
    'PART_QUANTITY': None,
    'MAKE': None,
    'MODEL': None,
    'MODEL_YEAR': None,
    'PLATE': None,
    'PLATE_STATE': None,
    'MANAGEMENT_SYSTEM': None,
    'LOCATION_ID': None,
    'LOCATION_NAME': 'provider_name',
    'ADDRESS': None,
    'CITY': 'provider_city',
    'STATE': 'provider_province',
    'POSTAL_CODE': None,
    'PHONE': None,
    'URL': None
}

This dictionary is my source-to-target mapping, defined as data instead of being hidden inside the logic. If a new column is added, I only need to add it here without changing the code. The E010 count comes from counting the entries with `None`, rather than from a separate rule.

In [17]:
mapped   = [c for c, f in shop_a_map.items() if f is not None]
unmapped = [c for c, f in shop_a_map.items() if f is None]
print(len(mapped), "mapped |", len(unmapped), "unmapped")
print(f'> Unmapped fields: {unmapped}')

9 mapped | 15 unmapped
> Unmapped fields: ['RO_CLOSE_DATE', 'LABOR_DESCRIPTION', 'PART_NAME_DESCRIPTION', 'PART_QUANTITY', 'MAKE', 'MODEL', 'MODEL_YEAR', 'PLATE', 'PLATE_STATE', 'MANAGEMENT_SYSTEM', 'LOCATION_ID', 'ADDRESS', 'POSTAL_CODE', 'PHONE', 'URL']


In [20]:
# this goes to tests/
print(f'> Dictionary and file match | no silent gaps: {set(shop_a_map) == set(shop_a_raw.columns)}')

> Dictionary and file match | no silent gaps: True


### 2.7 Profiling of the mapped columns

In [19]:
for source, target in shop_a_map.items():
    if target is not None:
        col = shop_a[source]
        print(f'\n> {source}')

        print(f'Nulls: {col.isna().sum()}')
        print(f'Unique: {col.nunique()}')

        if col.dtype == 'object':
            print(f'Trailing/leading spaces: {col.str.strip().ne(col).sum()}')


> VIN
Nulls: 0
Unique: 26
Trailing/leading spaces: 0

> RO_OPEN_DATE
Nulls: 0
Unique: 39
Trailing/leading spaces: 0

> MILEAGE
Nulls: 1
Unique: 40

> ODOMETER_MEASURE
Nulls: 0
Unique: 2
Trailing/leading spaces: 0

> RO_INVOICE_NUMBER
Nulls: 0
Unique: 41

> SERVICE_DESCRIPTION
Nulls: 1
Unique: 33
Trailing/leading spaces: 9

> LOCATION_NAME
Nulls: 0
Unique: 3
Trailing/leading spaces: 0

> CITY
Nulls: 0
Unique: 1
Trailing/leading spaces: 0

> STATE
Nulls: 0
Unique: 1
Trailing/leading spaces: 0


### 2.8 What the profiling forces
| Column | Finding | Rule required |
|---|---|---|
| MILEAGE | 1 null, thousands separator | Strip separator, nullable integer |
| ODOMETER_MEASURE | 2 units, no nulls | Convert MI to KM, keep source unit |
| RO_INVOICE_NUMBER | 41 unique of 42 | Exact duplicate, reject as E009 |
| SERVICE_DESCRIPTION | 1 null, 9 with spare spaces | Strip; a null description is E006 |
| LOCATION_NAME | 3 spellings of one shop | Normalize, or key on LOCATION_ID |
| VIN, CITY, STATE | clean | Copy; VIN validated at N.4, not here |

`strip` and `upper` on VIN are defensive, not driven by this profile: this feed is clean, and the rule exists because the next one may not be. `LOCATION_ID` has one value while `LOCATION_NAME` has three, so the ID is the reliable provider key and the name is display only.